# 02 - Prétraitement et sélection de variables (section 2.1)

Règle : tout ce qui apprend des paramètres (bornes IQR, moyenne et écart-type, sélection) est calculé sur le train uniquement. Le test n'influence jamais ces choix.

In [ ]:
import sys
from pathlib import Path

# Fonctionne qu'on lance le notebook depuis notebooks/ ou depuis la racine du projet
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

import json
import joblib
from collections import Counter

from sklearn.feature_selection import f_classif

from src.load_data import load_wine_data, make_binary_target, split_data
from src.preprocess import clean_data, build_preprocessor
from src.feature_selection import correlation_filter, select_kbest, select_rfe

## 1. Nettoyage, cible binaire et découpage train/test

In [ ]:
df = make_binary_target(clean_data(load_wine_data()))
X_train, X_test, y_train, y_test = split_data(df)

print("Train :", X_train.shape, "| proportion de bons :", round(y_train.mean(), 3))
print("Test  :", X_test.shape,  "| proportion de bons :", round(y_test.mean(), 3))

## 2. Plafonnement IQR et standardisation (ajustés sur le train)

In [ ]:
prep_all = build_preprocessor()
X_train_s = prep_all.fit_transform(X_train, y_train)
X_test_s = prep_all.transform(X_test)

X_train_s.describe().T[["mean", "std"]].round(3)

## 3. Filtre par corrélation

Une variable est redondante si sa corrélation avec une autre dépasse 0,8 en valeur absolue.

In [ ]:
to_drop = correlation_filter(X_train_s, threshold=0.8)
print("Variables redondantes (|r| > 0.8) :", to_drop)

## 4. SelectKBest (filtre univarié, F de l'ANOVA)

In [ ]:
K = 6

scores = pd.Series(f_classif(X_train_s, y_train)[0], index=X_train_s.columns)
print(scores.sort_values(ascending=False).round(1))

kbest = select_kbest(X_train_s, y_train, k=K)
print(f"\nSelectKBest (k = {K}) :", kbest)

## 5. RFE (élimination récursive, méthode wrapper)

In [ ]:
rfe = select_rfe(X_train_s, y_train, n_features=K)
print(f"RFE (n = {K}) :", rfe)

## 6. Choix final des variables

Règle retenue : une variable est gardée si les deux méthodes la sélectionnent.

In [ ]:
votes = Counter(kbest + rfe)
selected = [col for col, v in votes.items() if v >= 2]

print("Votes par variable :")
print(pd.Series(votes).sort_values(ascending=False))
print("\nVariables retenues :", selected)
print("Redondantes parmi les retenues :", [c for c in selected if c in to_drop])

**À compléter :** justifier la valeur de K, et expliquer le traitement d'une variable redondante qui serait retenue.

## 7. Pipeline final et sauvegarde

Le pipeline est ajusté sur le train. B et C l'importent et y ajoutent leur modèle.

In [ ]:
final_pipeline = build_preprocessor(selected).fit(X_train, y_train)

X_train_final = final_pipeline.transform(X_train)
X_test_final = final_pipeline.transform(X_test)
print("Forme après prétraitement :", X_train_final.shape, X_test_final.shape)
X_train_final.head()

In [ ]:
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(final_pipeline, MODELS_DIR / "preprocessor.joblib")
with open(MODELS_DIR / "selected_features.json", "w", encoding="utf-8") as f:
    json.dump(selected, f, indent=2, ensure_ascii=False)

print("Sauvegardé dans", MODELS_DIR, ":", sorted(p.name for p in MODELS_DIR.iterdir()))